In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
import torch_geometric.transforms as T
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/typing.py:47: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.27' not found (required by /home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/typing.py:101: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.27' not found (required by /home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [2]:
# Define the GAT model
# this implementation is credit to pytorch_geometric examples
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, data):
        h, edge_index = data.x, data.edge_index

        h = F.dropout(h, p=0.6, training=self.training)
        h = F.elu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.conv2(h, edge_index)

        return h

# Load the datasets
pubmed_dataset = Planetoid(root='/tmp/Pubmed', name='Pubmed', transform=T.NormalizeFeatures())
data = pubmed_dataset[0]
data = data.to(device)

Processing...
Done!


In [3]:
h_channels = 64
heads = 8
model = GAT(pubmed_dataset.num_features, h_channels, pubmed_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [4]:
h_channels = 64
heads = 8
model = GAT(pubmed_dataset.num_features, h_channels, pubmed_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [5]:
train(model, data, data.train_mask, data.y)

1.0993801355361938

In [6]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)

    acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    return acc

for epoch in range(0, 200):
    loss = train(model, data, data.train_mask, data.y)
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 1.0782, Accuracy: 0.1820
Epoch: 001, Loss: 1.0733, Accuracy: 0.4120
Epoch: 002, Loss: 1.0400, Accuracy: 0.5590
Epoch: 003, Loss: 1.0432, Accuracy: 0.6380
Epoch: 004, Loss: 1.0054, Accuracy: 0.6110
Epoch: 005, Loss: 0.9766, Accuracy: 0.6560
Epoch: 006, Loss: 0.9721, Accuracy: 0.7150
Epoch: 007, Loss: 0.9416, Accuracy: 0.7270
Epoch: 008, Loss: 0.9216, Accuracy: 0.7150
Epoch: 009, Loss: 0.9159, Accuracy: 0.7050
Epoch: 010, Loss: 0.9196, Accuracy: 0.6990
Epoch: 011, Loss: 0.8957, Accuracy: 0.6970
Epoch: 012, Loss: 0.8272, Accuracy: 0.6980
Epoch: 013, Loss: 0.8168, Accuracy: 0.7000
Epoch: 014, Loss: 0.7474, Accuracy: 0.7140
Epoch: 015, Loss: 0.7903, Accuracy: 0.7130
Epoch: 016, Loss: 0.7577, Accuracy: 0.7210
Epoch: 017, Loss: 0.6700, Accuracy: 0.7230
Epoch: 018, Loss: 0.6581, Accuracy: 0.7260
Epoch: 019, Loss: 0.6094, Accuracy: 0.7270
Epoch: 020, Loss: 0.6070, Accuracy: 0.7260
Epoch: 021, Loss: 0.6751, Accuracy: 0.7310
Epoch: 022, Loss: 0.5919, Accuracy: 0.7320
Epoch: 023,

In [7]:
torch.save(model.state_dict(), 'pubmed_gat.pt')